# **Практическое задание №1**

Установка необходимых пакетов:

In [ ]:
!pip install -q tqdm
!pip install --upgrade --no-cache-dir gdown
!pip install optuna

Монтирование Вашего Google Drive к текущему окружению:

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Импорт необходимых зависимостей:

In [ ]:
import random
from datetime import datetime
from pathlib import Path

import gdown
import matplotlib.pyplot as plt
import numpy as np
import optuna
import optuna.visualization as vis
import torch
import torchsummary
from optuna.samplers import TPESampler
from PIL import Image
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Subset
from torch.utils.data import Dataset as TorchDataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.models.efficientnet import EfficientNet_V2_S_Weights, efficientnet_v2_s
from torchvision.transforms import v2
from tqdm.notebook import tqdm

%load_ext tensorboard
%matplotlib inline

Константы, которые пригодятся в коде далее, и ссылки (gdrive идентификаторы) на предоставляемые наборы данных:

In [ ]:
EVALUATE_ONLY = False
TEST_ON_LARGE_DATASET = True
TISSUE_CLASSES = ('ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM')
NUM_CLASSES = len(TISSUE_CLASSES)
DATASETS_LINKS = {
    'train': '1XtQzVQ5XbrfxpLHJuL0XBGJ5U7CS-cLi',
    'train_small': '1qd45xXfDwdZjktLFwQb-et-mAaFeCzOR',
    'train_tiny': '1I-2ZOuXLd4QwhZQQltp817Kn3J0Xgbui',
    'test': '1RfPou3pFKpuHDJZ-D9XDFzgvwpUBFlDr',
    'test_small': '1wbRsog0n7uGlHIPGLhyN-PMeT2kdQ2lI',
    'test_tiny': '1viiB0s041CNsAK4itvX8PnYthJ-MDnQc'
}
SEED = 42

# Константы взяты из документации к моделям efficientnet_v2
NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

if not torch.cuda.is_available():
    print('CUDA is not available.  Training on CPU ...')
    DEVICE = torch.device("cpu")
else:
    print('CUDA is available!  Training on GPU ...')
    DEVICE = torch.device("cuda")

Подкачка всех .npz датасетов локально на диск

In [ ]:
# for name in DATASETS_LINKS:
#     url = f"https://drive.google.com/uc?export=download&confirm=pbef&id={DATASETS_LINKS[name]}"
#     output = f'/content/drive/MyDrive/{name}.npz'
#     gdown.download(url, output, quiet=False)

Зафиксируем `random_seed=42` для воспроизводимости

In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(SEED)

---
### Класс Dataset

Предназначен для работы с наборами данных, обеспечивает чтение изображений и соответствующих меток, а также формирование пакетов (батчей).

In [ ]:
class Dataset:

    def __init__(self, name):
        self.name = name
        self.is_loaded = False
        url = f"https://drive.google.com/uc?export=download&confirm=pbef&id={DATASETS_LINKS[name]}"

        # output = f'{name}.npz'
        # gdown.download(url, output, quiet=False)
        output = f'/content/drive/MyDrive/{name}.npz'
        gdown.download(url, output, quiet=False, skip_download=True)

        print(f'Loading dataset {self.name} from npz.')

        # np_obj = np.load(f'{name}.npz')
        np_obj = np.load(output)

        self.images = np_obj['data']
        self.labels = np_obj['labels']
        self.n_files = self.images.shape[0]
        self.is_loaded = True
        print(f'Done. Dataset {name} consists of {self.n_files} images.')

    def image(self, i):
        # read i-th image in dataset and return it as numpy array
        if self.is_loaded:
            return self.images[i, :, :, :]

    def images_seq(self, n=None):
        # sequential access to images inside dataset (is needed for testing)
        for i in range(self.n_files if not n else n):
            yield self.image(i)

    def random_image_with_label(self):
        # get random image with label from dataset
        i = np.random.randint(self.n_files)
        return self.image(i), self.labels[i]

    def random_batch_with_labels(self, n):
        # create random batch of images with labels (is needed for training)
        indices = np.random.choice(self.n_files, n)
        imgs = [self.image(i) for i in indices]
        labels = np.array([self.labels[i] for i in indices])
        return np.stack(imgs), labels

    def image_with_label(self, i: int):
        # return i-th image with label from dataset
        return self.image(i), self.labels[i]


### Обёртка над Dataset для использования с PyTorch

In [ ]:
try:
    class HistologyTorchDataset(TorchDataset):
        """
        Обёртка над Dataset для использования с PyTorch.

        base_dataset: экземпляр Dataset('train'), Dataset('train_small'), etc.
        transform:    функция/объект, преобразующий изображение (PIL.Image -> torch.Tensor).

        """
        def __init__(self, base_dataset: Dataset, mode: str):
            self.base = base_dataset
            self.mode = mode.lower()

            if self.mode not in {'train', 'eval'}:
                raise ValueError(f"{self.mode} is not correct; correct modes: {('train', 'eval')}")

            if self.mode == 'train':
                self.transform = v2.Compose([
                    v2.ToImage(),
                    v2.RandomHorizontalFlip(p=0.5),
                    v2.RandomVerticalFlip(p=0.5),
                    v2.ColorJitter(brightness=0.05, contrast=0.05),
                    v2.ToDtype(torch.float32, scale=True),
                    v2.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
                ])
            elif self.mode == 'eval':
                self.transform = v2.Compose([
                    v2.ToImage(),
                    v2.ToDtype(torch.float32, scale=True),
                    v2.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
                ])

        def __len__(self):
            # Размер датасета
            return len(self.base.images)

        def __getitem__(self, idx):
            """
            Возвращает (image_tensor, label) для PyTorch.
            image_tensor: torch.Tensor формы [3, H, W]
            label: int
            """
            img, label = self.base.image_with_label(idx)  # img: np.ndarray (H, W, 3)
            img = Image.fromarray(img)                    # в PIL.Image
            img = self.transform(img)     # в torch.Tensor
            return img, int(label)

        @property
        def labels(self):
            return self.base.labels

        @property
        def n_files(self):
            return self.base.n_files

except ImportError:
    HistologyTorchDataset = None
    print("PyTorch / torchvision не найдены. Обёртка HistologyTorchDataset недоступна.")


---
### Класс Metrics

Реализует метрики точности, используемые для оценивания модели:
1. точность,
2. сбалансированную точность.

In [ ]:
class Metrics:

    @staticmethod
    def accuracy(gt, pred):
        assert len(gt) == len(pred), 'gt and prediction should be of equal length'
        return sum(int(i[0] == i[1]) for i in zip(gt, pred)) / len(gt)

    @staticmethod
    def accuracy_balanced(gt, pred):
        return balanced_accuracy_score(gt, pred)

    @staticmethod
    def print_all(gt, pred, info: str):
        print(f'metrics for {info}:')
        print(f'\t accuracy {Metrics.accuracy(gt, pred):.4f}:')
        print(f'\t balanced accuracy {Metrics.accuracy_balanced(gt, pred):.4f}:')

---
### Класс Model

Класс, хранящий в себе всю информацию о модели.

Вам необходимо реализовать методы save, load для сохранения и загрузки модели. Особенно актуально это будет во время тестирования на дополнительных наборах данных.


> *Пожалуйста, убедитесь, что сохранение и загрузка модели работает корректно. Для этого обучите модель, протестируйте, сохраните ее в файл, перезапустите среду выполнения, загрузите обученную модель из файла, вновь протестируйте ее на тестовой выборке и убедитесь в том, что получаемые метрики совпадают с полученными для тестовой выборки ранее.*


Также, Вы можете реализовать дополнительные функции, такие как:
1. валидацию модели на части обучающей выборки;
2. использование кроссвалидации;
3. автоматическое сохранение модели при обучении;
4. загрузку модели с какой-то конкретной итерации обучения (если используется итеративное обучение);
5. вывод различных показателей в процессе обучения (например, значение функции потерь на каждой эпохе);
6. построение графиков, визуализирующих процесс обучения (например, график зависимости функции потерь от номера эпохи обучения);
7. автоматическое тестирование на тестовом наборе/наборах данных после каждой эпохи обучения (при использовании итеративного обучения);
8. автоматический выбор гиперпараметров модели во время обучения;
9. сохранение и визуализацию результатов тестирования;
10. Использование аугментации и других способов синтетического расширения набора данных (дополнительным плюсом будет обоснование необходимости и обоснование выбора конкретных типов аугментации)
11. и т.д.

Полный список опций и дополнений приведен в презентации с описанием задания.

При реализации дополнительных функций допускается добавление параметров в существующие методы и добавление новых методов в класс модели.


In [ ]:
class Model:
    BEST_MODEL_PATH = 'best_model'

    def __init__(self, model, writer=None, device=None):
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = model.to(self.device)
        self.num_classes = NUM_CLASSES
        self.writer = writer  # Для tensorboard

    def save(self, name: str, epoch, optimizer=None, scheduler=None):
        path = f'/content/drive/MyDrive/{name}.pth'
        ckpt = {
            'model_state': self.model.state_dict(),
            'arch': 'efficient_net_v2_s',
            'num_classes': self.num_classes,
            'optimizer_state': optimizer.state_dict() if optimizer else None,
            'scheduler_state': scheduler.state_dict() if scheduler else None,
            'epoch': epoch,
        }
        torch.save(ckpt, path)
        print(f'\nМодель сохранена в {path}.')
        print('Чтобы сдать файл лучшей модели ссылкой, откройте к нему доступ и вставьте id в load().')

    def load(self, name: str, optimizer=None, scheduler=None) -> int:
        # todo: если сдаёте модель ссылкой, замените None на id вашего файла.
        # Пока id не указан, читается файл, который save записал на примонтированный Drive.
        name_to_id_dict = {
            Model.BEST_MODEL_PATH: None,
        }
        file_id = name_to_id_dict.get(name)
        output = f'{name}.pth'
        if file_id:
            gdown.download(id=file_id, output=output, quiet=False)
            path = output
        else:
            path = f'/content/drive/MyDrive/{output}'
            if not Path(path).exists():
                raise FileNotFoundError(
                    f'Не найден {path}. Вызовите save("{name}") при смонтированном Google Drive '
                    f'или укажите id файла в name_to_id_dict внутри load().'
                )

        ckpt = torch.load(path, map_location=self.device, weights_only=False)
        self.model.load_state_dict(ckpt['model_state'])
        self.model.to(self.device)
        print(f'Модель загружена из {path}')

        if optimizer and ckpt.get('optimizer_state'):
            optimizer.load_state_dict(ckpt['optimizer_state'])
            print(f'Оптимизатор загружен из {path}')
        if scheduler and ckpt.get('scheduler_state'):
            scheduler.load_state_dict(ckpt['scheduler_state'])
            print(f'Шедулер загружен из {path}')
        return ckpt['epoch']

    def train(self, num_epochs, dataloaders, optimizer, loss_func, scheduler=None):
        info = {
            "valid_loss": [],
            "train_acc": [],
            "valid_acc": [],
            "train_acc_balanced": [],
            "valid_acc_balanced": []
        }
        best_acc = 0.0
        best_acc_balanced = 0.0
        best_score = 0.0
        PATIENCE = 4
        patience_counter = 0

        for epoch in range(num_epochs):
            print(f"\nEpoch: {epoch + 1}")

            train_preds, train_targets = self._fit_one_epoch(
                train_dataloader=dataloaders['train'],
                optimizer=optimizer,
                loss_func=loss_func,
                scheduler=scheduler
            )
            val_preds, val_targets, val_loss = self._eval_one_epoch(
                val_dataloader=dataloaders['valid'],
                loss_func=loss_func,
            )

            train_acc = Metrics.accuracy(train_targets, train_preds)
            print(f"\n{train_acc = :.5f}")
            valid_acc = Metrics.accuracy(val_targets, val_preds)
            print(f"{valid_acc = :.5f}")

            # Выводим значения
            train_acc_balanced = Metrics.accuracy_balanced(train_targets, train_preds)
            print(f"{train_acc_balanced = :.5f}")
            valid_acc_balanced = Metrics.accuracy_balanced(val_targets, val_preds)
            print(f"{valid_acc_balanced = :.5f}")

            # Добавляем информации для tensorboard
            if self.writer:
                self.writer.add_scalar('loss/valid', val_loss, epoch)
                self.writer.add_scalar('acc/train', train_acc, epoch)
                self.writer.add_scalar('acc/valid', valid_acc, epoch)
                self.writer.add_scalar('acc_balanced/train', train_acc_balanced, epoch)
                self.writer.add_scalar('acc_balanced/valid', valid_acc_balanced, epoch)

            info['valid_loss'].append(val_loss)
            info['train_acc'].append(train_acc)
            info['valid_acc'].append(valid_acc)
            info['train_acc_balanced'].append(train_acc_balanced)
            info['valid_acc_balanced'].append(valid_acc_balanced)

            score = 0.3 * valid_acc + 0.7 * valid_acc_balanced
            if score > best_score + 1e-4:
                patience_counter = 0
                best_acc = valid_acc
                best_acc_balanced = valid_acc_balanced
                best_score = 0.3 * best_acc + 0.7 * best_acc_balanced

                # Делаем лучший чекпоинт
                print(f" -> new {best_score = :.5f}, saved to {Model.BEST_MODEL_PATH}")
                self.save(Model.BEST_MODEL_PATH, epoch + 1, optimizer, scheduler)
            else:
                # Чекпоинт на случай, если что-то сломается и надо будет снова обучаться
                self.save('last_model', epoch + 1, optimizer, scheduler)

                patience_counter += 1
                if patience_counter >= PATIENCE:
                    print("Early Stopping! Модель переобучается!\n")
                    break
        return info

    def test_on_dataset(self, dataset: HistologyTorchDataset, batch_size=64, limit=None):
        self.model.eval()
        predictions = []

        if limit and 0 < limit <= 1:
            n = int(dataset.n_files * limit)
            subset = Subset(dataset, list(range(n)))
        else:
            n = dataset.n_files
            subset = dataset

        loader = DataLoader(
            subset,
            batch_size=batch_size,
            shuffle=False,
        )

        with torch.no_grad():
            for X, _ in tqdm(loader, desc='Testing...'):
                predictions.append(self.model(X.to(self.device)).argmax(dim=1).numpy(force=True))
        return np.concat(predictions)

    def test_on_image(self, img: np.ndarray, submission_year: int):
        self.model.eval()
        transform = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),
            v2.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD)
        ])

        tensor = transform(Image.fromarray(img)).unsqueeze(0).to(self.device)
        with torch.no_grad():
            outp = self.model(tensor)
            prediction = outp.argmax(dim=1)
        return prediction

    def _fit_one_epoch(self, train_dataloader, optimizer, loss_func, scheduler=None):
        self.model.train()

        all_preds = []
        all_targets = []

        for X_batch, y_batch in tqdm(train_dataloader, desc='Training...'):
            X_batch = X_batch.to(self.device)
            y_batch = y_batch.to(self.device)

            optimizer.zero_grad()

            outp = self.model(X_batch)
            loss = loss_func(outp, y_batch)

            loss.backward()
            optimizer.step()
            if scheduler:
                scheduler.step()

            preds = outp.argmax(dim=1)

            all_preds.append(preds.detach().cpu())
            all_targets.append(y_batch.detach().cpu())

        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        return all_preds, all_targets

    def _eval_one_epoch(self, val_dataloader, loss_func):
        self.model.eval()

        all_preds = []
        all_targets = []

        running_loss = 0.0
        total = 0

        for X_batch, y_batch in tqdm(val_dataloader, desc='Validation...'):
            with torch.no_grad():
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                outp = self.model(X_batch)
                loss = loss_func(outp, y_batch)

                preds = outp.argmax(dim=1)

                running_loss += loss.item() * X_batch.size(0)
                total += y_batch.size(0)

                all_preds.append(preds.detach().cpu())
                all_targets.append(y_batch.detach().cpu())

        epoch_loss = running_loss / total
        all_preds = torch.cat(all_preds).numpy()
        all_targets = torch.cat(all_targets).numpy()

        return all_preds, all_targets, epoch_loss

    def plot_training(self, info):
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(info['valid_loss'], label='valid_loss')
        axes[0].set_title('Loss')
        axes[0].legend()
        axes[0].set_xlabel('epoch')

        axes[1].plot(info['train_acc'], label='train_acc')
        axes[1].plot(info['valid_acc'], label='valid_acc')
        axes[1].plot(info['train_acc_balanced'], '--', label='train_bal')
        axes[1].plot(info['valid_acc_balanced'], '--', label='valid_bal')
        axes[1].set_title('Accuracy')
        axes[1].legend()
        axes[1].set_xlabel('epoch')

        plt.tight_layout()
        plt.show()

---
### Классификация изображений

Используя введенные выше классы можем перейти уже непосредственно к обучению модели классификации изображений. Пример общего пайплайна решения задачи приведен ниже. Вы можете его расширять и улучшать. В данном примере используются наборы данных 'train_small' и 'test_small'.

In [ ]:
base_train = Dataset('train_tiny')

train_idx, val_idx = train_test_split(
    np.arange(base_train.n_files),
    test_size=0.2,
    shuffle=True,
    random_state=SEED,
    stratify=base_train.labels,
)

train_ds = Subset(HistologyTorchDataset(base_train, mode='train'), train_idx)
val_ds = Subset(HistologyTorchDataset(base_train, mode='eval'), val_idx)

d_test = HistologyTorchDataset(Dataset('test_small'), mode='eval')

In [ ]:
batch_size = 400

Создаем train и valid DataLoaders. Для воспроизводимости результата добавлен seed_worker и generator,  
которые использует уже make_loader

In [ ]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(ds, batch_size, shuffle):
    kwargs = {
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': 2 if shuffle else 0,
    }
    if shuffle:
        kwargs['generator'] = torch.Generator().manual_seed(SEED)
        kwargs['worker_init_fn'] = seed_worker
    return DataLoader(ds, **kwargs)

train_dataloader = make_loader(train_ds, batch_size, shuffle=True)
valid_dataloader = make_loader(val_ds, batch_size, shuffle=False)

dataloaders = {
    'train': train_dataloader,
    'valid': valid_dataloader,
}

Создание модели вида `efficientnet_v2`

In [ ]:
def build_model(n, w, layers_unfreeze=2, dropout=0.2):
    m = n(weights=w)

    for p in m.features[:-layers_unfreeze].parameters():
        p.requires_grad = False

    num_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=dropout, inplace=True),
        nn.Linear(num_features, NUM_CLASSES),
    )
    print(f"Number of classes = {NUM_CLASSES}")

    m.to(DEVICE)
    return m

Подбор гиперпараметров на основе TPE

In [ ]:
# def objective(trial):
#     # Гиперпараметры
#     lr = trial.suggest_float('lr', 1e-6, 3e-3, log=True)
#     max_lr = trial.suggest_float('max_lr', 1e-5, 1e-3, log=True)
#     weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-2, log=True)

#     seed_everything(SEED)

#     # --- лоадеры под текущий batch_size ---
#     train_loader = make_loader(train_ds, batch_size, shuffle=True)
#     val_loader = make_loader(val_ds, batch_size, shuffle=False)

#     dataloaders = {
#         'train': train_loader,
#         'valid': val_loader,
#     }

#     # --- новая модель и оптимизатор на каждый trial ---
#     net = build_model(efficientnet_v2_s, EfficientNet_V2_S_Weights.IMAGENET1K_V1)
#     m = Model(net, device=DEVICE)
#     num_epochs = 5
#     optimizer = torch.optim.AdamW(
#         [p for p in m.model.parameters() if p.requires_grad],
#         lr=lr,
#         weight_decay=weight_decay,
#     )
#     crit = nn.CrossEntropyLoss()
#     sched = torch.optim.lr_scheduler.OneCycleLR(
#         optimizer,
#         max_lr=max_lr,
#         epochs=num_epochs,  # Короткое обучение для сравнения
#         steps_per_epoch=len(train_loader),
#     )

#     # Обучение
#     m.train(5, dataloaders, optimizer, crit, sched)

#     # Метрика на валидации
#     val_preds, val_targets, _ = m._eval_one_epoch(val_loader, crit)
#     score = Metrics.accuracy_balanced(val_targets, val_preds)

#     # Очистка памяти
#     del m, net, optimizer, train_loader, val_loader
#     torch.cuda.empty_cache()

#     return float(score)

Запуск

In [ ]:
# study = optuna.create_study(
#     direction='maximize',
#     sampler=TPESampler(seed=SEED),
# )
# n_trials = 10
# study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

# # Запомнив параметры, можно захардкодить их и обучить модель на них
# print('Best params:', study.best_params)
# print('Best balanced accuracy:', study.best_value)

In [ ]:
# vis.plot_optimization_history(study).show()
# vis.plot_param_importances(study).show()
# vis.plot_parallel_coordinate(study).show()

Теперь обучимся на лучших гиперпараметрах

In [ ]:
seed_everything(SEED)
eff_net_s = build_model(efficientnet_v2_s, EfficientNet_V2_S_Weights.IMAGENET1K_V1, dropout=0.2)

Теперь введем все переменные, которые нам нужны

In [ ]:
train_val_writer = SummaryWriter('runs/nnip_prac_1_run')
num_epochs = 30
batches_per_epoch = len(train_dataloader)

model = Model(model=eff_net_s, writer=train_val_writer, device=DEVICE)
optimizer = torch.optim.AdamW(model.model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=1e-3,
    epochs=num_epochs,
    steps_per_epoch=batches_per_epoch,
)

In [ ]:
if not EVALUATE_ONLY:
    info = model.train(
        num_epochs,
        dataloaders,
        optimizer,
        criterion,
        scheduler
    )

Нарисуем графики, которые были построены в train

In [ ]:
%tensorboard --logdir runs

In [ ]:
model.plot_training(info)

Подгрузка лучшей версии модели

In [ ]:
model.load('best_model')

Пример тестирования модели на части набора данных:

In [ ]:
# # evaluating model on 10% of test dataset
# pred_1 = model.test_on_dataset(d_test, limit=0.1)
# Metrics.print_all(d_test.labels[:len(pred_1)], pred_1, '10% of test')

Пример тестирования модели на полном наборе данных:

In [ ]:
# evaluating model on full test dataset (may take time)
if TEST_ON_LARGE_DATASET:
    pred_2 = model.test_on_dataset(d_test, batch_size)
    Metrics.print_all(d_test.labels, pred_2, 'test')

Результат работы пайплайна обучения и тестирования выше тоже будет оцениваться. Поэтому не забудьте присылать на проверку ноутбук с выполнеными ячейками кода с демонстрациями метрик обучения, графиками и т.п. В этом пайплайне Вам необходимо продемонстрировать работу всех реализованных дополнений, улучшений и т.п.

<font color="red">
Настоятельно рекомендуется после получения пайплайна с полными результатами обучения экспортировать ноутбук в pdf (файл -> печать) и прислать этот pdf вместе с самим ноутбуком.
</font>

### Тестирование модели на других наборах данных

Ваша модель должна поддерживать тестирование на других наборах данных. Для удобства, Вам предоставляется набор данных test_tiny, который представляет собой малую часть (2% изображений) набора test. Ниже приведен фрагмент кода, который будет осуществлять тестирование для оценивания Вашей модели на дополнительных тестовых наборах данных.

<font color="red">
Прежде чем отсылать задание на проверку, убедитесь в работоспособности фрагмента кода ниже.
</font>


In [ ]:
final_model = Model(model=eff_net_s, device=DEVICE)
final_model.load('best_model')
d_test_tiny = HistologyTorchDataset(Dataset('test_tiny'), mode='eval')
pred = final_model.test_on_dataset(d_test_tiny)
Metrics.print_all(d_test_tiny.labels, pred, 'test-tiny')

Google Drive пока не отключаем: примеры архивов ниже пишут файлы на Диск. Отключение стоит в последней ячейке.


---
# Дополнительные "полезности"

Ниже приведены примеры использования различных функций и библиотек, которые могут быть полезны при выполнении данного практического задания.

### Измерение времени работы кода

Измерять время работы какой-либо функции можно легко и непринужденно при помощи функции timeit из соответствующего модуля:

In [ ]:
import timeit


def factorial(n):
    res = 1
    for i in range(1, n + 1):
        res *= i
    return res


def f():
    return factorial(n=1000)

n_runs = 128
print(f'Function f is caluclated {n_runs} times in {timeit.timeit(f, number=n_runs)}s.')

<font color="red">
Для эффективной работы с моделями глубокого обучения убедитесь в том, что в текущей среде Google Colab используется аппаратный ускоритель GPU или TPU. Для смены среды выберите "среда выполнения" -> "сменить среду выполнения".
</font>

### PyTorch

Готовой сети и цикла обучения в шаблоне нет: это часть задания. Данные для PyTorch берёт класс `HistologyTorchDataset` выше. Ниже — ссылки на документацию.


### Дополнительные ресурсы по PyTorch

- **Официальные туториалы PyTorch** — https://pytorch.org/tutorials/  
- **“Deep Learning with PyTorch: 60-Minute Blitz”** — https://pytorch.org/tutorials/beginner/deep_learning_60min_blitz.html  
- **Transfer Learning for Computer Vision** — https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html  
- **PyTorch Get Started (установка)** — https://pytorch.org/get-started/locally/  
- **Dive into Deep Learning (D2L, глава PyTorch)** — https://d2l.ai/chapter_preliminaries/index.html  
- **Fast.ai — Practical Deep Learning for Coders** — https://course.fast.ai/  
- **Learn PyTorch.io (Zero to Mastery)** — https://www.learnpytorch.io/

Отмонтировать Google Drive.


In [ ]:
# drive.flush_and_unmount()
